In [1]:
import numpy as np
import torch
import hydra
from hydra import initialize, compose
from omegaconf import DictConfig

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict

from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

In [2]:
from os.path import join, dirname
import os
import sys

from einops import rearrange, repeat

sys.path.append(join(dirname(dirname(os.getcwd()))))

from memory_perceiver.main.configuration import cf

from memory_perceiver.core.model_net import PerceiverIO
from memory_perceiver.processing.vectorize import input_to_tensor, fourier_encode

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
nb_frames = 32
frames_future = 8
len_video_frame = 1824  # 456*256/(8*8)
len_audio_frame = 120   # 1920/16
query_dim = 832
video_latent_dim = 256
audio_latent_dim = 560
num_workers = 2
batch_size = 4

len_vid = nb_frames*len_video_frame
len_audio = nb_frames*len_audio_frame
len_audio_raw = len_audio*cf.sample_per_frame
video_future = frames_future*len_video_frame
audio_future = frames_future*len_audio_frame

special_compression = [i for i in range(video_future)] + [i for i in range(len_vid, len_vid+audio_future)]
for k in range(len(cf.body_parts)):
    special_compression += [ len_vid + len_audio + k*nb_frames + i for i in range(frames_future) ]

In [5]:
def create_model(device=device):

    # Projection to return modalities, enhancement

    modality_parameters = {}
    modality_parameters['audio'] = audio_latent_dim
    modality_parameters['video'] = video_latent_dim
    for part, params in cf.body_parts.items():
        modality_parameters[part] = query_dim - 2*128 - params

    projection_parameters = dict(cf.body_parts)
    projection_parameters.update({'audio': 16})

    # Instantiate model
        
    model = PerceiverIO(
            2,
            query_dim,
            special_compression = special_compression,
            device=device,
            projection_parameters = projection_parameters,
            modality_parameters = modality_parameters,
            max_batch_size = 4
        )
    model = model.to(device)

    # Queries

    queries = []
    audio_query = repeat( torch.cat([model.modality['audio'], torch.zeros(16, device=device)]), '... -> n ...', n=audio_future)
    audio_pos = fourier_encode(torch.linspace(-1, 1, audio_future), audio_future/2, num_bands=128).to(device)
    queries.append(torch.cat([audio_query, audio_pos], dim=-1).to(device))
    for part, params in cf.body_parts.items():
        part_query = repeat( torch.cat([model.modality[part], torch.zeros(params, device=device)]), '... -> n ...', n=frames_future)
        part_pos = fourier_encode(torch.linspace(-1, 1, frames_future), frames_future/2, num_bands=128).to(device)
        queries.append(torch.cat([part_query, part_pos], dim=-1).to(device))
    queries=torch.cat(queries)


    return model, queries

In [6]:
model, queries = create_model()

In [7]:
model.modality.keys()

dict_keys(['audio', 'video', 'observations'])

In [8]:
inputs={}
inputs['video']=torch.randn(4, 3, 32, 256, 456, device=device)
inputs['audio']=torch.randn(4, 32*1920, device=device)
inputs['observations']=torch.randn(4, 32, 4, device=device)

In [9]:
batch = input_to_tensor(inputs, model.modality, cf.body_parts).to(device)

In [10]:
with torch.no_grad():
    x, aux_loss = model(batch, queries = queries)

In [11]:
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):

    cfg_dict = omegaconf_to_dict(cfg)
    print_dict(cfg_dict)

    headless = cfg.headless
    render = not headless

    env = VecEnvRLGames(headless=headless, sim_device=cfg.device_id)
    # sets seed. if seed is -1 will pick a random one
    from omni.isaac.core.utils.torch.maths import set_seed
    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed
    task = initialize_task(cfg_dict, env)

    save_step = 160
    obs_buffer = []
    reward_buffer = []
    action_buffer = []
    while env._simulation_app.is_running():
        if env._world.is_playing():
            if env._world.current_time_step_index == 0:
                env._world.reset(soft=True)

            # 'cartpole_view' 'obs_buf' 512, 4
            obs = env._task.get_observations()
            reward = env._task.calculate_metrics()
            print('obs: ', obs, '\n', '$$$', '\n') 

            
            # obs_buffer.append(obs)
            # if(len(obs_buffer)>32):
                #  obs_buffer.pop(0)
            # reward_buffer.append(reward)
            # if(len(reward_buffer)>32):
                #  reward_buffer.pop(0)
            # media['video'] = default_video
            # media['audio'] = default_audio
            #
            # if env.sim_frame_count % save_step == 0 and len(obs_buffer) >= save_step:
            # Save mem body, audio, video


            # if(len(action_buffer) == 0 and len(obs_buffer) >= 8):
                # obs.append(get camera view())
                # obs.append(audio)
                # with torch.no_grad():
                    # action, voice = memory_perceiver(obs_buffer[-8:])
                # action_buffer.append(action)
            # if(len(action_buffer)>0):
                # action = action_buffer.pop(0)
            # else:
                # action = torch.randn(512, 1)


            action = torch.tensor(np.array([env.action_space.sample() for _ in range(env.num_envs)]), device=task.rl_device)
            # 512, 1
            print('action: ', action.size(), '\n', '$$$', '\n') 
            env._task.pre_physics_step(action)
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()
        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Cartpole", "headless=True"])
    # print(OmegaConf.to_yaml(cfg))
    parse_hydra_configs(cfg)
    # run_with_config(parse_hydra_configs)


/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


task: 
    name: Cartpole
    physics_engine: physx
    env: 
        numEnvs: 512
        envSpacing: 4.0
        resetDist: 3.0
        maxEffort: 400.0
        clipObservations: 5.0
        clipActions: 1.0
        controlFrequencyInv: 2
    sim: 
        dt: 0.0083
        use_gpu_pipeline: True
        gravity: [0.0, 0.0, -9.81]
        add_ground_plane: True
        add_distant_light: True
        use_flatcache: True
        enable_scene_query_support: False
        disable_contact_processing: False
        enable_cameras: False
        default_physics_material: 
            static_friction: 1.0
            dynamic_friction: 1.0
            restitution: 0.0
        physx: 
            worker_thread_count: 4
            solver_type: 1
            use_gpu: True
            solver_position_iteration_count: 4
            solver_velocity_iteration_count: 0
            contact_offset: 0.02
            rest_offset: 0.001
            bounce_threshold_velocity: 0.2
            friction_of

2023-02-22 07:09:30 s] [Error] [carb.graphics-vulkan.plugin] VkResult: ERROR_INITIALIZATION_FAILED
2023-02-22 07:09:30 s] [Error] [carb.graphics-vulkan.plugin] vkCreateInstance failed. Vulkan 1.1 is not supported, or your driver requires an update.
2023-02-22 07:09:30 s] [Error] [gpu.foundation.plugin] carb::graphics::createInstance failed.
2023-02-22 07:09:30 s] [Error] [carb.graphics-vulkan.plugin] VkResult: ERROR_INITIALIZATION_FAILED
2023-02-22 07:09:30 s] [Error] [carb.graphics-vulkan.plugin] vkCreateInstance failed. Vulkan 1.1 is not supported, or your driver requires an update.
2023-02-22 07:09:30 s] [Error] [gpu.foundation.plugin] carb::graphics::createInstance failed.
2023-02-22 07:09:30 s] [Error] [omni.gpu_foundation_factory.plugin] Failed to create GPU foundation devices for compatibilityMode!


[0.591s] [ext: omni.isaac.core_archive-2.0.1] startup
[0.606s] [ext: omni.pip.torch-1_13_0-0.1.4] startup
[0.613s] [ext: omni.isaac.ml_archive-1.1.0] startup
[0.614s] [ext: omni.client-0.1.1] startup
[0.630s] [ext: omni.appwindow-1.0.1] startup
2023-02-22 07:09:31 s] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-02-22 07:09:31 s] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
[0.638s] [ext: omni.kit.renderer.core-0.0.0] startup
[0.645s] [ext: omni.kit.renderer.capture-0.0.0] startup
[0.650s] [ext: omni.kit.renderer.imgui-0.0.0] startup
2023-02-22 07:09:31 s] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-02-22 07:09:31 s] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glf

2023-02-22 07:09:31 s] [Error] [omni.kit.renderer.plugin] GPU Foundation is not initialized!
2023-02-22 07:09:31 s] [Error] [omni.kit.imgui_renderer.plugin] _prepareDefaultWindow: invalid device attached to the default window!


[0.978s] [ext: omni.usd.schema.anim-0.0.0] startup
[1.137s] [ext: omni.usd.schema.omnigraph-1.0.0] startup
[1.156s] [ext: omni.gpucompute.plugins-0.0.0] startup
[1.156s] [ext: omni.hydra.scene_delegate-0.3.0] startup
[1.165s] [ext: omni.hydra.usdrt_delegate-4.1.1] startup
[1.188s] [ext: omni.usdphysics-104.1.6-5.1] startup
[1.191s] [ext: omni.kit.window.popup_dialog-2.0.15] startup
[1.203s] [ext: omni.kit.actions.core-1.0.0] startup
[1.209s] [ext: omni.kit.audiodeviceenum-1.0.0] startup
[1.213s] [ext: omni.kit.widget.nucleus_connector-1.0.2] startup
[1.221s] [ext: omni.kit.commands-1.4.5] startup
[1.231s] [ext: omni.kvdb-0.0.0] startup
[1.239s] [ext: omni.convexdecomposition-104.1.6-5.1] startup
[1.247s] [ext: omni.usd.core-1.0.0] startup
[1.250s] [ext: omni.timeline-1.0.5] startup
[1.256s] [ext: omni.kit.numpy.common-0.1.0] startup
[1.261s] [ext: omni.usd-1.6.23] startup
[1.329s] [ext: omni.localcache-0.0.0] startup
[1.336s] [ext: omni.isaac.version-1.0.0] startup
[1.338s] [ext: omni.

2023-02-22 07:09:32 [1,710ms] [Error] [carb.graphics-vulkan.plugin] VkResult: ERROR_INITIALIZATION_FAILED
2023-02-22 07:09:32 [1,710ms] [Error] [carb.graphics-vulkan.plugin] vkCreateInstance failed. Vulkan 1.1 is not supported, or your driver requires an update.
2023-02-22 07:09:32 [1,710ms] [Error] [gpu.foundation.plugin] carb::graphics::createInstance failed.
2023-02-22 07:09:32 [1,710ms] [Error] [omni.gpu_foundation_factory.plugin] Failed to create GPU foundation devices!


: 

: 